In [114]:
import torch
import timeit


ten = torch.randn((2,3,5,6), device="meta")
# conv = torch.conv2d(3, 5,2, device="meta")
conv = torch.nn.Conv2d(3, 5, 2, 2, device="meta")
start = timeit.default_timer()
out = conv(ten)
end = timeit.default_timer()

print(out.size(), f"{end-start:.6f}")


torch.Size([2, 5, 2, 3]) 0.000744


In [115]:
t = torch.rand(3**10, 3, 5**10, 6, device="meta")
conv = torch.nn.Conv2d(3, 5,2, 2, device="meta")
start = timeit.default_timer()
out = conv(t)
end = timeit.default_timer()

print(out.size(), f"{end - start:.6f}")

torch.Size([59049, 5, 4882812, 3]) 0.000356


In [117]:
from torch import nn
import torch.nn.functional as F

class my_conv(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(3,6,2)
        self.maxp1 = nn.MaxPool2d(2,2)
        self.conv2 = nn.Conv2d(6,12,3)
        self.fc1 = nn.Linear(12 * 12 * 35,60)
        self.fc2 = nn.Linear(60,30)
        self.fc3 = nn.Linear(30,10)

    def forward(self, x):
        x = self.conv1(x)
        x = self.maxp1(F.relu(x))
        x = self.maxp1(F.relu(self.conv2(x)))
        x = torch.flatten(x, 1)
        x = self.fc1(x)
        x = self.fc2(x)
        x = self.fc3(x)
        return x

In [118]:
def forward_hook(module, input, output):
    print(f"shape of output of module {module} is {output.shape}")

with torch.device("meta"):
    model = my_conv()
    inp = torch.rand(2**10, 3, 2**7, 2**6)

for name, layer in model.named_modules():
    layer.register_forward_hook(forward_hook)

out = model(inp)

shape of output of module Conv2d(3, 6, kernel_size=(2, 2), stride=(1, 1)) is torch.Size([1024, 6, 127, 63])
shape of output of module MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False) is torch.Size([1024, 6, 63, 31])
shape of output of module Conv2d(6, 12, kernel_size=(3, 3), stride=(1, 1)) is torch.Size([1024, 12, 61, 29])
shape of output of module MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False) is torch.Size([1024, 12, 30, 14])
shape of output of module Linear(in_features=5040, out_features=60, bias=True) is torch.Size([1024, 60])
shape of output of module Linear(in_features=60, out_features=30, bias=True) is torch.Size([1024, 30])
shape of output of module Linear(in_features=30, out_features=10, bias=True) is torch.Size([1024, 10])
shape of output of module my_conv(
  (conv1): Conv2d(3, 6, kernel_size=(2, 2), stride=(1, 1))
  (maxp1): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (conv2): Conv2d(6, 12